In [37]:
import os

In [38]:
%pwd

'd:\\'

In [41]:
os.chdir('..')

In [47]:
%cd "D:\PredictBot-Score-MLOps"

D:\PredictBot-Score-MLOps


In [48]:
from dataclasses import dataclass 
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.constants import CONFIG_PATH
from pathlib import Path
import pandas as pd

In [49]:
@dataclass(frozen=True)
class DataValidationConfig:

    # --- paths ---
    raw_data_path   : Path
    validated_data  : Path
    quarantine_data : Path
    log_path        : Path

    # --- step 1 : schema ---
    expected_columns : list[str]
    expected_dtypes  : dict[str, str]

    # --- step 2 : data quality ---
    missing_thresholds      : dict[str, float]
    allow_duplicate_timestamps : bool
    volume                  : dict[str, int]

    # --- step 3 : temporal ---
    temporal : dict[str, object]

    # --- step 4 : statistical ---
    statistical : dict[str, float]

    # --- step 5 : feature validation ---
    range_checks  : dict[str, dict]
    outlier_floor : float
    max_step_drop : float

    # --- step 6 : business logic ---
    business_logic : dict[str, object]

    # --- outcome routing ---
    hard_fail_checks : list[str]
    soft_fail_checks : list[str]

In [50]:
class config_manager:

    def __init__(self,config = CONFIG_PATH):
        self.config = yaml_load(config)
        
        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self):
        config = self.config.data_validation

        create_directories([config.validated_data , config.quarantine_data])
        
        data_validation_config = DataValidationConfig(
            raw_data_path=Path(config.raw_data_folder),
            validated_data=Path(config.validated_data),
            quarantine_data=Path(config.quarantine_data),
            log_path=Path(config.log_path),
            
            expected_columns=config.expected_columns,
            expected_dtypes=config.expected_dtypes,
            
            missing_thresholds=config.missing_thresholds,
            allow_duplicate_timestamps=config.duplicate_timestamps,
            volume=config.volume,
            
            temporal=config.temporal,
            statistical=config.statistical,
            
            range_checks=config.range_checks,
            outlier_floor=config.outlier_floor,
            max_step_drop=config.max_step_drop,
            
            business_logic=config.business_logic,
            hard_fail_checks=config.hard_fail_checks,
            soft_fail_checks=config.soft_fail_checks
        )
        
        return data_validation_config

In [51]:
xc = config_manager()
xc.get_data_validation_config()

[2026-06-09 19:33:25,985: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-09 19:33:25,991: INFO: common: Directory created (or already exists) at: artifacts]
[2026-06-09 19:33:25,994: INFO: common: Directory created (or already exists) at: artifacts/data_validation/validated/]
[2026-06-09 19:33:25,997: INFO: common: Directory created (or already exists) at: artifacts/data_validation/quarantine/]


DataValidationConfig(raw_data_path=WindowsPath('artifacts/data_ingestion/raw_data'), validated_data=WindowsPath('artifacts/data_validation/validated'), quarantine_data=WindowsPath('artifacts/data_validation/quarantine'), log_path=WindowsPath('artifacts/data_validation/logs/log.json'), expected_columns=BoxList(['timestamp', 'bot_score']), expected_dtypes=ConfigBox({'timestamp': 'datetime64[ns, UTC]', 'bot_score': 'float64'}), missing_thresholds=ConfigBox({'timestamp': 0.0, 'bot_score': 0.0}), allow_duplicate_timestamps=False, volume=ConfigBox({'min_rows': 96, 'max_rows': 50000, 'expected': 40705}), temporal=ConfigBox({'interval_minutes': 15, 'max_gap_count': 0, 'require_sorted': True}), statistical=ConfigBox({'baseline_mean': 0.81, 'baseline_std': 0.09, 'drift_factor': 2.0, 'min_std': 0.001}), range_checks=ConfigBox({'bot_score': {'min': 0.0, 'max': 1.0}}), outlier_floor=0.5, max_step_drop=0.2, business_logic=ConfigBox({'allow_cold_start': True, 'known_anomaly_score': 0.0, 'max_identica

In [ ]:
class Data_validation:

    def __init__(self, config : DataValidationConfig):
        self.config = config

    def read_data(self):
        try:
            logger.info(f"Attempting to read data from: {self.config.raw_data_path}")
            raw_data = pd.read_csv(self.config.raw_data_path)
            logger.info("Data successfully loaded into memory.")
            return raw_data
        
        except FileNotFoundError:
            # This is your custom message
            print("--- ALERT: The file is missing! Please check the path. ---")
            
            # This is the log file entry
            logger.error(f"File not found at: {self.config.raw_data_path}")
            
            # This re-throws the error so the pipeline stops safely
            raise 

    def validate_null_values(self):
        try:
            data = self.read_data()
            logger.info("Starting Null-value validation...")
            
            # Check for any null values
            if data.isnull().values.any():
                # Count them to give a helpful log message
                null_counts = data.isnull().sum()
                logger.error(f"NULL VALUES FOUND:\n{null_counts[null_counts > 0]}")
                
                raise ValueError("Dataset contains null values. Validation failed.")
            
            logger.info("SUCCESS: No null values found in the dataset.")
            return True
            
        except Exception as e:
            logger.error(f"Null validation aborted: {str(e)}")
            raise
            
        except Exception as e:

            logger.error(f"Validation process aborted: {str(e)}")
            raise


    def validate_volume(self):
    
        try:
            data = self.read_data()
            min_rows = self.config.volume.min_rows
            max_rows = self.config.volume.max_rows
            row_count = len(data)
            
            logger.info(f"Starting Volume validation: {row_count} rows found.")
            
            # 1. Check Minimum
            if row_count < min_rows:
                logger.error(f"VOLUME TOO LOW: Expected at least {min_rows} rows, but found {row_count}.")
                raise ValueError(f"Dataset too small: {row_count} rows.")
                
            # 2. Check Maximum (Safety Ceiling)
            if row_count > max_rows:
                logger.warning(f"VOLUME ABNORMALLY HIGH: Expected max {max_rows} rows, but found {row_count}.")
                # Usually we log a warning but continue, or we can choose to fail here
                # raise ValueError(f"Dataset too large: {row_count} rows.")
            
            logger.info(f"SUCCESS: Volume validation passed ({row_count} rows).")
            return True
            
        except Exception as e:
            logger.error(f"Volume validation aborted: {str(e)}")
            raise
    
    def validate_temporal_integrity(self):
        """
        Validates the time-series structure: interval, gaps, and sorting.
        """
        try:
            data = self.read_data()
            # Ensure timestamp is datetime type
            data['timestamp'] = pd.to_datetime(data['timestamp'])
            
            logger.info("Starting Temporal Integrity validation...")
            
            # 1. Check if sorted
            if self.config.temporal.require_sorted:
                if not data['timestamp'].is_monotonic_increasing:
                    logger.error("TEMPORAL ERROR: Timestamp column is not sorted.")
                    raise ValueError("Data is not sorted chronologically.")
            
            # 2. Check intervals and gaps
            # Calculate the difference between consecutive timestamps
            diffs = data['timestamp'].diff().dropna()
            expected_interval = pd.Timedelta(minutes=self.config.temporal.interval_minutes)
            
            # Find gaps that don't match the expected interval
            gaps = diffs[diffs != expected_interval]
            
            if len(gaps) > self.config.temporal.max_gap_count:
                logger.error(f"TEMPORAL ERROR: Found {len(gaps)} gaps that do not match {expected_interval} interval.")
                raise ValueError(f"Temporal gaps detected: {len(gaps)} violations.")
            
            logger.info("SUCCESS: Temporal integrity validation passed.")
            return True
            
        except Exception as e:
            logger.error(f"Temporal validation aborted: {str(e)}")
            raise 
    
    def validate_statistical_drift(self):
        """
        Validates the statistical distribution of the bot_score column.
        """
        try:
            data = self.read_data()
            stats = self.config.statistical
            
            logger.info("Starting Statistical Drift validation...")
            
            # Calculate current stats
            current_mean = data['bot_score'].mean()
            current_std = data['bot_score'].std()
            
            # 1. Check for suspicious lack of variance (are all values identical?)
            if current_std < stats.min_std:
                logger.error(f"STATISTICAL ANOMALY: Variance too low ({current_std}). Data might be corrupted.")
                raise ValueError("Suspiciously identical values detected in bot_score.")
            
            # 2. Check for Drift (Did the mean shift significantly?)
            # Logic: |current_mean - baseline_mean| > (drift_factor * baseline_std)
            drift_threshold = stats.drift_factor * stats.baseline_std
            if abs(current_mean - stats.baseline_mean) > drift_threshold:
                logger.error(f"DATA DRIFT DETECTED: Mean {current_mean:.4f} shifted beyond threshold from {stats.baseline_mean:.4f}")
                raise ValueError("Data drift detected: statistical distribution has changed.")
            
            logger.info(f"SUCCESS: Statistical validation passed. Mean: {current_mean:.4f}, Std: {current_std:.4f}")
            return True
            
        except Exception as e:
            logger.error(f"Statistical validation aborted: {str(e)}")
            raise
    

    



    
        
        
        


In [160]:
con = config_manager()
con_data_injestion = con.get_data_validation_config()
con_data_injestion = Data_validation(con_data_injestion)
con_data_injestion.validate_statistical_drift()


[2026-06-09 23:56:12,143: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-09 23:56:12,153: INFO: common: Directory created (or already exists) at: artifacts]


[2026-06-09 23:56:12,161: INFO: common: Directory created (or already exists) at: artifacts/data_validation/validated/]
[2026-06-09 23:56:12,165: INFO: common: Directory created (or already exists) at: artifacts/data_validation/quarantine/]
[2026-06-09 23:56:12,170: INFO: 3376309460: Attempting to read data from: artifacts\data_ingestion\raw_data\Cloud_fare_Master_data.csv]
[2026-06-09 23:56:12,381: INFO: 3376309460: Data successfully loaded into memory.]
[2026-06-09 23:56:12,384: INFO: 3376309460: Starting Statistical Drift validation...]
[2026-06-09 23:56:12,432: INFO: 3376309460: SUCCESS: Statistical validation passed. Mean: 0.8100, Std: 0.0903]


True